# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all dataset elements by `@id`.

### Dataset Source
The dataset Croissant schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define URL of the Croissant schema
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
md = dataset.metadata
print(f"Title: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Version: {md.version}")

## 2. Data Overview
Review available record sets and fields in the dataset.

Entities are always referenced by their `@id`.

In [ ]:
# List all record sets with their `@id` and name
print('Available record sets:')
for rs in md.record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use `@id`.

In [ ]:
# Choose a record set by its @id
# For demonstration, select the first record set
record_set_ids = [rs.id for rs in md.record_sets]
print(f"Available record set @ids: {record_set_ids}")
chosen_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

if chosen_record_set_id and not dataframes[chosen_record_set_id].empty:
    print(f"Columns in record set {{chosen_record_set_id}}: ")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print('No records found in the chosen record set.')

## 4. Exploratory Data Analysis (EDA)
This section demonstrates data processing using the field `@id`s.

- Filtering numeric values
- Normalization
- Grouping (if categorical field exists)

In [ ]:
# Use the first numeric field available for demonstration
if chosen_record_set_id and not dataframes[chosen_record_set_id].empty:
    record_set = next(rs for rs in md.record_sets if rs.id == chosen_record_set_id)
    numeric_field_id = None
    for field in record_set.fields:
        # Accept ints/floats per Croissant schema type
        if field.data_type in ["schema:Float", "schema:Integer", "schema:Number"]:
            numeric_field_id = field.id
            break

    if numeric_field_id and numeric_field_id in dataframes[chosen_record_set_id].columns:
        print(f"Numeric field selected: {numeric_field_id}")
        df = dataframes[chosen_record_set_id]
        # Try parsing column as float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} :")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field if present
        cat_field_id = None
        for field in record_set.fields:
            if field.data_type in ["schema:Text", "schema:Boolean"] and field.id != numeric_field_id:
                cat_field_id = field.id
                break
        if cat_field_id and cat_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(cat_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {cat_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print('No records or no valid record set available.')

## 5. Visualization
Visualize distributions or relationships between fields using `matplotlib` and `seaborn` if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (
    chosen_record_set_id
    and not dataframes[chosen_record_set_id].empty
    and numeric_field_id
    and numeric_field_id in dataframes[chosen_record_set_id].columns
):
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[chosen_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    # Try categorical field
    if cat_field_id and cat_field_id in dataframes[chosen_record_set_id].columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=cat_field_id, y=numeric_field_id, data=dataframes[chosen_record_set_id])
        plt.title(f"{numeric_field_id} by {cat_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No suitable data for visualization.')

## 6. Conclusion
In this notebook, we loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`, reviewed its record sets and fields (referencing all by `@id`), loaded records into DataFrames, performed exploratory analysis, and visualized key data distributions. All manipulations referenced each dataset element by `@id`, ensuring traceability and schema alignment.

For deeper analyses, consult the dataset's Croissant schema for further field semantics and metadata.